# Phase 0 - Synthetic Dataset Generation

This notebook creates the datasets requested for the capstone project: customer, product, order, order-item, and clickstream data, including deliberate data quality issues and day-2 schema evolution.

In [1]:
from pathlib import Path
import csv
import json
import random
from datetime import datetime, timedelta

random.seed(2026)
base_dir = Path.cwd()
data_dir = base_dir / 'data'
data_dir.mkdir(exist_ok=True)


def write_jsonl(path, records):
    with path.open('w', encoding='utf-8') as fh:
        for record in records:
            fh.write(json.dumps(record) + '\n')


customer_count = 5000
unique_customer_ids = [f'CUS{i:06d}' for i in range(1, 4951)]
customer_ids = unique_customer_ids.copy()
for _ in range(50):
    customer_ids.append(random.choice(unique_customer_ids))

regions = ['North', 'South', 'East', 'West', 'Central']
customer_records = []
for idx, customer_id in enumerate(customer_ids, start=1):
    if random.random() < 0.02:
        email = ''
    elif random.random() < 0.02:
        email = 'not-an-email'
    else:
        email = f'{customer_id.lower()}@example.com'
    customer_records.append({
        'customer_id': customer_id,
        'name': f'Customer {idx}',
        'email': email,
        'signup_date': (datetime(2021, 1, 1) + timedelta(days=random.randint(0, 1200))).strftime('%Y-%m-%d'),
        'region': random.choice(regions),
        'is_active': random.choice([True, False])
    })

customer_fieldnames = ['customer_id', 'name', 'email', 'signup_date', 'region', 'is_active']
with (data_dir / 'customers.csv').open('w', newline='', encoding='utf-8') as fh:
    writer = csv.DictWriter(fh, fieldnames=customer_fieldnames)
    writer.writeheader()
    writer.writerows(customer_records)

product_categories = ['Electronics', 'Home', 'Sports', 'Books', 'Beauty', 'Garden']
product_records = []
for i in range(1, 801):
    product_records.append({
        'product_id': f'PRD{i:05d}',
        'category': random.choice(product_categories),
        'unit_price': round(random.uniform(5, 250), 2),
        'active_flag': random.choice([True, False])
    })
product_ids = [record['product_id'] for record in product_records]
product_fieldnames = ['product_id', 'category', 'unit_price', 'active_flag']
with (data_dir / 'products.csv').open('w', newline='', encoding='utf-8') as fh:
    writer = csv.DictWriter(fh, fieldnames=product_fieldnames)
    writer.writeheader()
    writer.writerows(product_records)

order_records_day1 = []
for i in range(1, 20001):
    order_records_day1.append({
        'order_id': f'ORD{i:08d}',
        'customer_id': random.choice(customer_ids),
        'order_ts': (datetime(2024, 1, 1) + timedelta(hours=random.randint(0, 20000))).strftime('%Y-%m-%dT%H:%M:%SZ'),
        'store_region': random.choice(regions),
        'status': random.choice(['completed', 'pending', 'cancelled', 'returned'])
    })
write_jsonl(data_dir / 'orders_day1.json', order_records_day1)

order_item_records_day1 = []
for order in order_records_day1:
    item_count = random.choices([2, 3, 4], weights=[0.6, 0.3, 0.1])[0]
    for _ in range(item_count):
        quantity = random.choices([1, 2, 3, 4], weights=[0.8, 0.1, 0.05, 0.05])[0]
        if random.random() < 0.01:
            quantity = 0
        elif random.random() < 0.01:
            quantity = -1
        if random.random() < 0.01:
            product_id = 'PRD99999'
        else:
            product_id = random.choice(product_ids)
        product_row = next((p for p in product_records if p['product_id'] == product_id), None)
        unit_price = product_row['unit_price'] if product_row else round(random.uniform(5, 100), 2)
        line_total = round(quantity * unit_price, 2)
        order_item_records_day1.append({
            'order_id': order['order_id'],
            'product_id': product_id,
            'quantity': quantity,
            'unit_price': unit_price,
            'line_total': line_total
        })
write_jsonl(data_dir / 'order_items_day1.json', order_item_records_day1)

discount_codes = ['SAVE10', 'WELCOME', 'SPRING20', 'VIP15', 'NONE']
order_records_day2 = []
for i in range(1, 4001):
    order_records_day2.append({
        'order_id': f'ORD2{i:08d}',
        'customer_id': random.choice(customer_ids),
        'store_region': random.choice(regions),
        'status': random.choice(['completed', 'pending', 'cancelled', 'returned']),
        'discount_code': random.choice(discount_codes),
        'order_ts': (datetime(2024, 1, 2) + timedelta(hours=random.randint(0, 12000))).strftime('%Y-%m-%dT%H:%M:%SZ')
    })
write_jsonl(data_dir / 'orders_day2.json', order_records_day2)

order_item_records_day2 = []
for order in order_records_day2:
    item_count = random.choices([2, 3, 4], weights=[0.6, 0.3, 0.1])[0]
    for _ in range(item_count):
        quantity = random.choices([1, 2, 3, 4], weights=[0.8, 0.1, 0.05, 0.05])[0]
        if random.random() < 0.01:
            quantity = 0
        elif random.random() < 0.01:
            quantity = -1
        if random.random() < 0.01:
            product_id = 'PRD99999'
        else:
            product_id = random.choice(product_ids)
        product_row = next((p for p in product_records if p['product_id'] == product_id), None)
        unit_price = product_row['unit_price'] if product_row else round(random.uniform(5, 100), 2)
        line_total = round(quantity * unit_price, 2)
        order_item_records_day2.append({
            'order_id': order['order_id'],
            'product_id': product_id,
            'quantity': quantity,
            'unit_price': unit_price,
            'line_total': line_total,
            'discount_code': order['discount_code']
        })
write_jsonl(data_dir / 'order_items_day2.json', order_item_records_day2)

clickstream_day1 = []
for i in range(15000):
    clickstream_day1.append({
        'event_id': f'EVT{i:08d}',
        'customer_id': random.choice(customer_ids),
        'session_id': f'SESS{i:08d}',
        'event_ts': (datetime(2024, 1, 1, 0, 0) + timedelta(minutes=random.randint(0, 60000))).strftime('%Y-%m-%dT%H:%M:%SZ'),
        'page': random.choice(['home', 'product', 'cart', 'checkout']),
        'event_type': random.choice(['view', 'click', 'purchase'])
    })
write_jsonl(data_dir / 'clickstream_day1.json', clickstream_day1)

clickstream_day2 = []
for i in range(15000):
    clickstream_day2.append({
        'event_id': f'EVT2{i:08d}',
        'customer_id': random.choice(customer_ids),
        'session_id': f'SESS2{i:08d}',
        'event_ts': (datetime(2024, 1, 2, 0, 0) + timedelta(minutes=random.randint(0, 60000))).strftime('%Y-%m-%dT%H:%M:%SZ'),
        'page': random.choice(['home', 'product', 'cart', 'checkout']),
        'event_type': random.choice(['view', 'click', 'purchase'])
    })
write_jsonl(data_dir / 'clickstream_day2.json', clickstream_day2)

print('Generated files:')
for file_path in sorted(data_dir.glob('*')):
    print(file_path.name)
print('Customer rows:', len(customer_records))
print('Product rows:', len(product_records))
print('Orders day1 rows:', len(order_records_day1))
print('Order items day1 rows:', len(order_item_records_day1))
print('Orders day2 rows:', len(order_records_day2))
print('Order items day2 rows:', len(order_item_records_day2))
print('Clickstream day1 rows:', len(clickstream_day1))
print('Clickstream day2 rows:', len(clickstream_day2))

Generated files:
clickstream_day1.json
clickstream_day2.json
customers.csv
order_items_day1.json
order_items_day2.json
orders_day1.json
orders_day2.json
products.csv
Customer rows: 5000
Product rows: 800
Orders day1 rows: 20000
Order items day1 rows: 49983
Orders day2 rows: 4000
Order items day2 rows: 10085
Clickstream day1 rows: 15000
Clickstream day2 rows: 15000
